In [ ]:
pip install langchain_huggingface langchain_pinecone langchain_groq

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")
secret_value_1 = user_secrets.get_secret("GROQ_API_KEY")
secret_value_2 = user_secrets.get_secret("PINECONE_API_KEY")

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
from functools import lru_cache
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any

from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Optional: for semantic chunking (for future experiments)
try:
    from langchain_experimental.text_splitter import SemanticChunker
    SEMANTIC_AVAILABLE = True
except ImportError:
    SEMANTIC_AVAILABLE = False

# Import Kaggle Secrets
from kaggle_secrets import UserSecretsClient

# =========================================
# LOAD SECRETS FROM KAGGLE
# =========================================
print("=" * 80)
print("🔐 LOADING API KEYS FROM KAGGLE SECRETS")
print("=" * 80)

try:
    user_secrets = UserSecretsClient()
    
    GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    print("✓ GROQ_API_KEY loaded successfully")
    
    PINECONE_API_KEY = user_secrets.get_secret("PINECONE_API_KEY")
    print("✓ PINECONE_API_KEY loaded successfully")
    
    PINECONE_API_KEY_PHRAMACY = user_secrets.get_secret("PINECONE_API_KEY_PHRAMACY")
    print("✓ PINECONE_API_KEY_PHRAMACY loaded successfully")
    
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
    
except Exception as e:
    print(f"⚠️ Error loading secrets: {e}")
    load_dotenv()
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "false"

# =========================================
# CONFIGURATION
# =========================================
SIMILARITY_THRESHOLD = 0.50
MAX_CHUNK_LENGTH = 2000
K_RETRIEVAL = 5
K_RETRIEVE_FOR_RERANK = 10  # Retrieve more documents for reranking

GROQ_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
PINECONE_INDEX_NAME = "dermatologist"

# Number of questions to sample from each namespace
N_DIAGNOSIS_QUESTIONS = 10
N_MEDICATION_QUESTIONS = 10

# Reranker Configuration
USE_RERANKER = True  # Set to False to disable reranking
BGE_RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  # Best BGE reranker

# =========================================
# EMBEDDING MODELS
# =========================================
print("\n🔬 Loading Embedding Models...")

# Using MedEmbed for better medical embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="abhinand/MedEmbed-base-v0.1",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("✓ MedEmbed-base-v0.1 loaded")

similarity_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device='cpu')
print("✓ BAAI/bge-large-en-v1.5 loaded")

# =========================================
# BGE RERANKER INITIALIZATION - FIXED
# =========================================
print("\n🔄 Initializing BGE Reranker...")
reranker = None  # CRITICAL FIX: Initialize globally
try:
    reranker = CrossEncoder(BGE_RERANKER_MODEL)
    print(f"✓ BGE Reranker loaded: {BGE_RERANKER_MODEL}")
    USE_RERANKER = True  # Enable if loaded successfully
except Exception as e:
    print(f"⚠️ Failed to load BGE Reranker: {e}")
    reranker = None
    USE_RERANKER = False
    print("   → Reranking will be disabled")

# =========================================
# IMPROVED: Semantic Chunker with MedEmbed
# =========================================
def get_semantic_chunker():
    """Initialize SemanticChunker with MedEmbed for future experiments"""
    if SEMANTIC_AVAILABLE:
        try:
            # Use the same MedEmbed model for semantic chunking
            chunker = SemanticChunker(
                embedding_model,  # Using MedEmbed
                breakpoint_threshold_type="percentile"
            )
            print("✓ SemanticChunker ready (MedEmbed)")
            return chunker
        except Exception as e:
            print(f"⚠️ Could not initialize SemanticChunker: {e}")
    return None

semantic_chunker = get_semantic_chunker()

# =========================================
# ANSWER GENERATION PROMPT (CRITICAL FIX)
# =========================================
ANSWER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["question", "context"],
    template="""You are a dermatology expert. Answer the following QUESTION based ONLY on the provided CONTEXT. 

CRITICAL RULES:
1. ONLY use information from the CONTEXT below
2. DO NOT add any external knowledge or hallucinate
3. If the context doesn't contain the answer, say "The provided context does not contain information about this"
4. Be specific and medically accurate
5. Use complete sentences

QUESTION: {question}

CONTEXT:
{context}

ANSWER (based ONLY on the context above):"""
)

# =========================================
# FAITHFULNESS EVALUATION PROMPT (IMPROVED)
# =========================================
FAITHFULNESS_PROMPT = PromptTemplate(
    input_variables=["question", "answer", "context"],
    template="""You are a medical RAG evaluation expert. Evaluate if the ANSWER is FAITHFULLY grounded in the CONTEXT.

QUESTION: {question}

CONTEXT:
{context}

ANSWER:
{answer}

INSTRUCTIONS:
- Check EACH claim in the answer against the context
- Score = (number of claims supported by context) / (total claims in answer)
- If answer says "context does not contain information", score = 1.0
- Be strict: any information not in context = hallucination

Output ONLY JSON:
{{
    "faithfulness_score": 0.0-1.0,
    "reasoning": "Brief explanation",
    "supported_claims": ["claim1", "claim2"],
    "unsupported_claims": ["claim3"]
}}"""
)

# =========================================
# ANSWER CORRECTNESS PROMPT
# =========================================
ANSWER_CORRECTNESS_PROMPT = PromptTemplate(
    input_variables=["question", "answer", "ground_truth"],
    template="""You are a medical RAG evaluation expert. Compare the ANSWER to the GROUND TRUTH.

QUESTION: {question}

GROUND TRUTH (correct answer):
{ground_truth}

ANSWER (generated from RAG):
{answer}

INSTRUCTIONS:
- Score based on semantic similarity and factual overlap
- 1.0 = perfectly correct, 0.0 = completely wrong

Output ONLY JSON:
{{
    "correctness_score": 0.0-1.0,
    "reasoning": "Brief explanation"
}}"""
)

# =========================================
# CONTEXT PRECISION PROMPT
# =========================================
CONTEXT_PRECISION_PROMPT = PromptTemplate(
    input_variables=["question", "context"],
    template="""You are a medical RAG evaluation expert. Evaluate how PRECISE the CONTEXT is for answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}

INSTRUCTIONS:
- Score = proportion of context that is directly relevant to the question
- 1.0 = all context relevant, 0.0 = no context relevant

Output ONLY JSON:
{{
    "context_precision_score": 0.0-1.0,
    "reasoning": "Brief explanation"
}}"""
)

# =========================================
# ANSWER RELEVANCE PROMPT
# =========================================
ANSWER_RELEVANCE_PROMPT = PromptTemplate(
    input_variables=["question", "answer"],
    template="""Evaluate how RELEVANT the ANSWER is to the QUESTION.

QUESTION: {question}

ANSWER: {answer}

INSTRUCTIONS:
- Does the answer directly address the question?
- 1.0 = perfectly relevant, 0.0 = completely irrelevant

Output ONLY JSON:
{{
    "answer_relevance_score": 0.0-1.0,
    "reasoning": "Brief explanation"
}}"""
)

# =========================================
# CONTEXT RECALL PROMPT
# =========================================
CONTEXT_RECALL_PROMPT = PromptTemplate(
    input_variables=["ground_truth", "context"],
    template="""Evaluate RECALL - how much GROUND TRUTH information is covered in the CONTEXT.

GROUND TRUTH:
{ground_truth}

CONTEXT:
{context}

INSTRUCTIONS:
- Score = percentage of ground truth information found in context
- 1.0 = all ground truth covered, 0.0 = none covered

Output ONLY JSON:
{{
    "context_recall_score": 0.0-1.0,
    "reasoning": "Brief explanation"
}}"""
)

# =========================================
# LLM EVALUATOR CLASS
# =========================================
class LLMEvaluator:
    def __init__(self, model_name=GROQ_MODEL):
        self.llm = ChatGroq(
            model=model_name,
            temperature=0,
            max_tokens=1000,
            groq_api_key=GROQ_API_KEY
        )
        print(f"✓ LLM Evaluator initialized with: {model_name}")
    
    def _parse_json_response(self, response: str) -> dict:
        try:
            import re
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            return {}
        except Exception as e:
            print(f"      JSON parsing error: {e}")
            return {}
    
    def generate_answer(self, question: str, context: str) -> str:
        """Generate answer based ONLY on context"""
        prompt = ANSWER_GENERATION_PROMPT.format(
            question=question,
            context=context[:7000]  # INCREASED from 4000 to 7000 for better context
        )
        response = self.llm.invoke(prompt)
        return response.content.strip()
    
    def evaluate_faithfulness(self, question: str, answer: str, context: str) -> dict:
        prompt = FAITHFULNESS_PROMPT.format(
            question=question,
            answer=answer,
            context=context[:3000]
        )
        response = self.llm.invoke(prompt)
        return self._parse_json_response(response.content)
    
    def evaluate_answer_correctness(self, question: str, answer: str, ground_truth: str) -> dict:
        prompt = ANSWER_CORRECTNESS_PROMPT.format(
            question=question,
            answer=answer,
            ground_truth=ground_truth[:2000]
        )
        response = self.llm.invoke(prompt)
        return self._parse_json_response(response.content)
    
    def evaluate_context_precision(self, question: str, context: str) -> dict:
        prompt = CONTEXT_PRECISION_PROMPT.format(
            question=question,
            context=context[:3000]
        )
        response = self.llm.invoke(prompt)
        return self._parse_json_response(response.content)
    
    def evaluate_answer_relevance(self, question: str, answer: str) -> dict:
        prompt = ANSWER_RELEVANCE_PROMPT.format(
            question=question,
            answer=answer
        )
        response = self.llm.invoke(prompt)
        return self._parse_json_response(response.content)
    
    def evaluate_context_recall(self, ground_truth: str, context: str) -> dict:
        prompt = CONTEXT_RECALL_PROMPT.format(
            ground_truth=ground_truth[:2000],
            context=context[:3000]
        )
        response = self.llm.invoke(prompt)
        return self._parse_json_response(response.content)
    
    def evaluate_all(self, question: str, answer: str, ground_truth: str, context: str) -> dict:
        results = {}
        
        print(f"    • Evaluating faithfulness...")
        results['faithfulness'] = self.evaluate_faithfulness(question, answer, context)
        
        print(f"    • Evaluating answer correctness...")
        results['answer_correctness'] = self.evaluate_answer_correctness(question, answer, ground_truth)
        
        print(f"    • Evaluating context precision...")
        results['context_precision_llm'] = self.evaluate_context_precision(question, context)
        
        print(f"    • Evaluating answer relevance...")
        results['answer_relevance'] = self.evaluate_answer_relevance(question, answer)
        
        print(f"    • Evaluating context recall...")
        results['context_recall_llm'] = self.evaluate_context_recall(ground_truth, context)
        
        return results

# =========================================
# VECTOR STORES & RETRIEVAL
# =========================================
def get_store(namespace):
    return PineconeVectorStore(
        index_name=PINECONE_INDEX_NAME,
        embedding=embedding_model,
        namespace=namespace
    )

def retrieve_with_reranking(query, namespace, k=K_RETRIEVAL, use_reranker=True):
    """
    Retrieve documents with optional BGE reranking
    """
    # Step 1: Initial vector search (retrieve more for reranking)
    retrieve_k = K_RETRIEVE_FOR_RERANK if use_reranker else k
    retriever = get_store(namespace).as_retriever(
        search_kwargs={"k": retrieve_k}
    )
    docs = retriever.invoke(query)
    
    print(f"   📄 Retrieved {len(docs)} documents from vector search")
    
    # Step 2: Apply BGE Reranking if enabled
    if use_reranker and reranker is not None and len(docs) > 1:
        try:
            print(f"   🔄 Applying BGE Reranker...")
            # Prepare pairs for reranking
            pairs = [[query, doc.page_content] for doc in docs]
            scores = reranker.predict(pairs)
            
            # Sort documents by reranker score (higher is better)
            scored_docs = list(zip(docs, scores))
            scored_docs.sort(key=lambda x: x[1], reverse=True)
            
            # Keep top-k documents
            reranked_docs = []
            for doc, score in scored_docs[:k]:
                doc.metadata['rerank_score'] = float(score)
                doc.metadata['rerank_type'] = 'bge'
                reranked_docs.append(doc)
            
            print(f"   ✅ Reranked to {len(reranked_docs)} documents")
            
            # Show reranking scores
            for i, doc in enumerate(reranked_docs, 1):
                score = doc.metadata.get('rerank_score', 'N/A')
                preview = doc.page_content[:100].replace('\n', ' ')
                print(f"   Doc {i} (score: {score:.4f}): {preview}...")
            
            return reranked_docs
            
        except Exception as e:
            print(f"   ⚠️ Reranking failed: {e}, using original ranking")
            return docs[:k]
    
    return docs[:k]

def retrieve(query, namespace, k=K_RETRIEVAL):
    """Legacy retrieval function without reranking"""
    retriever = get_store(namespace).as_retriever(
        search_kwargs={"k": k}
    )
    return retriever.invoke(query)

# =========================================
# SEMANTIC SIMILARITY
# =========================================
@lru_cache(maxsize=1000)
def _cached_encode_bge(text: str):
    if len(text) > MAX_CHUNK_LENGTH:
        text = text[:MAX_CHUNK_LENGTH]
    return similarity_model.encode([text], normalize_embeddings=True)

def semantic_similarity(text1: str, text2: str) -> float:
    emb1 = _cached_encode_bge(text1)
    emb2 = _cached_encode_bge(text2)
    return float(np.dot(emb1[0], emb2[0]))

def is_relevant(ground_truth: str, doc, threshold: float = SIMILARITY_THRESHOLD) -> bool:
    return semantic_similarity(ground_truth, doc.page_content) >= threshold

# =========================================
# METRICS
# =========================================
def context_recall_continuous(ground_truth: str, docs) -> float:
    if len(docs) == 0:
        return 0.0
    return float(max(semantic_similarity(ground_truth, doc.page_content) for doc in docs))

def context_precision_continuous(ground_truth: str, docs) -> float:
    if len(docs) == 0:
        return 0.0
    return float(np.mean([semantic_similarity(ground_truth, doc.page_content) for doc in docs]))

def mrr_score(ground_truth: str, docs, threshold: float = SIMILARITY_THRESHOLD) -> float:
    for rank, doc in enumerate(docs, start=1):
        if semantic_similarity(ground_truth, doc.page_content) >= threshold:
            return 1.0 / rank
    return 0.0

def ndcg_score(ground_truth: str, docs) -> float:
    if len(docs) == 0:
        return 0.0
    scores = [semantic_similarity(ground_truth, doc.page_content) for doc in docs]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(scores))
    ideal_scores = sorted(scores, reverse=True)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_scores))
    return float(dcg / idcg if idcg > 0 else 0.0)

def precision_at_k(ground_truth: str, docs, k) -> float:
    if len(docs) == 0 or k == 0:
        return 0.0
    top_k = docs[:min(k, len(docs))]
    relevant = sum(1 for doc in top_k if is_relevant(ground_truth, doc))
    return float(relevant / k)

def recall_at_k(ground_truth: str, docs, k) -> float:
    if len(docs) == 0:
        return 0.0
    top_k = docs[:min(k, len(docs))]
    return float(1.0 if any(is_relevant(ground_truth, doc) for doc in top_k) else 0.0)

# =========================================
# COMBINE CONTEXT FROM DOCUMENTS
# =========================================
def combine_context(docs, max_chars=7000):  # INCREASED from 4000 to 7000
    """Combine retrieved documents into a single context string"""
    context_parts = []
    for i, doc in enumerate(docs, 1):
        part = f"[Document {i}]:\n{doc.page_content[:1200]}\n"  # INCREASED from 800 to 1200
        context_parts.append(part)
        
        if len('\n'.join(context_parts)) > max_chars:
            break
    
    return '\n'.join(context_parts)

# =========================================
# PLOTTING FUNCTIONS
# =========================================
def create_plots(results_df, llm_results, summary):
    """Create comprehensive visualization plots"""
    
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    
    fig = plt.figure(figsize=(20, 16))
    
    # Plot 1: Overall Metrics
    ax1 = fig.add_subplot(3, 3, 1)
    metrics = ['Recall', 'Precision', 'MRR', 'NDCG']
    scores = [
        summary['overall_metrics']['Context Recall (continuous)'],
        summary['overall_metrics']['Context Precision (continuous)'],
        summary['overall_metrics']['MRR'],
        summary['overall_metrics']['NDCG']
    ]
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(metrics)))
    bars = ax1.bar(metrics, scores, color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_ylim(0, 1.05)
    ax1.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax1.set_title('📊 Retrieval Metrics', fontsize=13, fontweight='bold')
    for bar, score in zip(bars, scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{score:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Plot 2: LLM Metrics
    ax2 = fig.add_subplot(3, 3, 2)
    llm_metrics = ['Faithfulness', 'Correctness', 'Context\nPrecision', 'Answer\nRelevance', 'Context\nRecall']
    llm_scores = [
        llm_results['avg_faithfulness'],
        llm_results['avg_correctness'],
        llm_results['avg_context_precision_llm'],
        llm_results['avg_answer_relevance'],
        llm_results['avg_context_recall_llm']
    ]
    bars2 = ax2.bar(llm_metrics, llm_scores, color=plt.cm.plasma(np.linspace(0, 0.8, 5)), 
                    edgecolor='black', linewidth=1.5)
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax2.set_title('🤖 LLM Metrics (RAGAS)', fontsize=13, fontweight='bold')
    ax2.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
    ax2.legend()
    for bar, score in zip(bars2, llm_scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{score:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Plot 3: Precision@K
    ax3 = fig.add_subplot(3, 3, 3)
    k_values = [1, 2, 3, 4, 5]
    precision_k = [summary['overall_metrics'][f'Precision@{k}'] for k in k_values]
    ax3.plot(k_values, precision_k, 'o-', linewidth=2, markersize=8, color='#2E86AB')
    ax3.fill_between(k_values, precision_k, alpha=0.2, color='#2E86AB')
    ax3.set_xlabel('k', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Precision@k', fontsize=11, fontweight='bold')
    ax3.set_title('📈 Precision@K', fontsize=13, fontweight='bold')
    ax3.set_xticks(k_values)
    ax3.set_ylim(0, 1.05)
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Recall@K
    ax4 = fig.add_subplot(3, 3, 4)
    recall_k = [summary['overall_metrics'][f'Recall@{k}'] for k in k_values]
    ax4.plot(k_values, recall_k, 's-', linewidth=2, markersize=8, color='#A23B72')
    ax4.fill_between(k_values, recall_k, alpha=0.2, color='#A23B72')
    ax4.set_xlabel('k', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Recall@k', fontsize=11, fontweight='bold')
    ax4.set_title('📈 Recall@K', fontsize=13, fontweight='bold')
    ax4.set_xticks(k_values)
    ax4.set_ylim(0, 1.05)
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Similarity Distribution
    ax5 = fig.add_subplot(3, 3, 5)
    best_sims = results_df['best_similarity'].values
    ax5.hist(best_sims, bins=20, color='#06A77D', edgecolor='black', alpha=0.7)
    ax5.axvline(x=np.mean(best_sims), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(best_sims):.3f}')
    ax5.set_xlabel('Similarity Score', fontsize=11, fontweight='bold')
    ax5.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax5.set_title('📊 Similarity Distribution', fontsize=13, fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Namespace Comparison
    ax6 = fig.add_subplot(3, 3, 6)
    diag_scores = results_df[results_df['namespace'] == 'diagnosis_knowledge']['best_similarity'].values
    med_scores = results_df[results_df['namespace'] == 'medication_knowledge']['best_similarity'].values
    bp = ax6.boxplot([diag_scores, med_scores], labels=['Diagnosis\n(20 queries)', 'Medication\n(20 queries)'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#95B8D1')
    bp['boxes'][1].set_facecolor('#D19592')
    ax6.set_ylabel('Best Similarity', fontsize=11, fontweight='bold')
    ax6.set_title('📦 Performance by Namespace (20 each)', fontsize=13, fontweight='bold')
    ax6.grid(True, alpha=0.3)
    
    # Plot 7: Heatmap
    ax7 = fig.add_subplot(3, 3, 7)
    top_queries = results_df.head(30)[['recall_continuous', 'precision_continuous', 'mrr', 'ndcg_graded']].values
    im = ax7.imshow(top_queries.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax7.set_yticks(range(4))
    ax7.set_yticklabels(['Recall', 'Precision', 'MRR', 'NDCG'])
    ax7.set_xlabel('Query Index', fontsize=11, fontweight='bold')
    ax7.set_title('🔥 Per-Query Heatmap', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax7, label='Score')
    
    # Plot 8: LLM Distribution
    ax8 = fig.add_subplot(3, 3, 8)
    llm_data = [
        llm_results['faithfulness_scores'],
        llm_results['correctness_scores'],
        llm_results['context_precision_scores'],
        llm_results['answer_relevance_scores']
    ]
    bp2 = ax8.boxplot(llm_data, positions=[1,2,3,4], widths=0.6, patch_artist=True)
    colors_box = ['#4ECDC4', '#FFE66D', '#FF6B6B', '#C7B9FF']
    for patch, color in zip(bp2['boxes'], colors_box):
        patch.set_facecolor(color)
    ax8.set_xticks([1,2,3,4])
    ax8.set_xticklabels(['Faithfulness', 'Correctness', 'Context\nPrecision', 'Answer\nRelevance'])
    ax8.set_ylabel('Score', fontsize=11, fontweight='bold')
    ax8.set_title('📊 LLM Score Distribution', fontsize=13, fontweight='bold')
    ax8.set_ylim(0, 1.05)
    ax8.axhline(y=0.7, color='green', linestyle='--', alpha=0.5)
    ax8.grid(True, alpha=0.3)
    
    # Plot 9: Radar Chart
    ax9 = fig.add_subplot(3, 3, 9, projection='polar')
    categories = ['Recall', 'Precision', 'MRR', 'Faithfulness', 'Correctness', 'Context Prec']
    values = [
        summary['overall_metrics']['Context Recall (continuous)'],
        summary['overall_metrics']['Context Precision (continuous)'],
        summary['overall_metrics']['MRR'],
        llm_results['avg_faithfulness'],
        llm_results['avg_correctness'],
        llm_results['avg_context_precision_llm']
    ]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]
    ax9.plot(angles, values, 'o-', linewidth=2, color='#2E86AB')
    ax9.fill(angles, values, alpha=0.25, color='#2E86AB')
    ax9.set_xticks(angles[:-1])
    ax9.set_xticklabels(categories, fontsize=9)
    ax9.set_ylim(0, 1)
    ax9.set_title('🎯 Radar Chart', fontsize=13, fontweight='bold', pad=20)
    ax9.grid(True)
    
    plt.tight_layout()
    plt.savefig('evaluation_plots.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Saved plots to evaluation_plots.png")
    
    return fig

# =========================================
# MAIN EVALUATION LOOP
# =========================================
print("=" * 80)
print("🧪 DERMATOLOGY RAG SYSTEM EVALUATION")
print(f"📅 Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

print("\n🔬 Model Configuration:")
print(f"  • LLM: GROQ {GROQ_MODEL}")
print(f"  • K: {K_RETRIEVAL}")
print(f"  • Threshold: {SIMILARITY_THRESHOLD}")
print(f"  • Diagnosis Questions: {N_DIAGNOSIS_QUESTIONS}")
print(f"  • Medication Questions: {N_MEDICATION_QUESTIONS}")
print(f"  • BGE Reranker: {'✅ ENABLED' if USE_RERANKER else '❌ DISABLED'}")
print(f"  • Reranker Model: {BGE_RERANKER_MODEL}")
print(f"  • Context Budget: 7000 chars (increased from 4000)")
print(f"  • Document Preview: 1200 chars (increased from 800)")

if not GROQ_API_KEY:
    print("\n❌ ERROR: GROQ_API_KEY not found!")
    sys.exit(1)

print("\nLoading dataset...")
dataset = pd.read_csv("/kaggle/input/datasets/salmankey/dermatology-ragas22-dataset-22/dermatology_ragas_dataset (1).csv")
print(f"✓ Loaded {len(dataset)} total queries")

# =========================================
# SELECT 20 DIAGNOSIS + 20 MEDICATION QUERIES
# =========================================
print(f"\n📋 Selecting {N_DIAGNOSIS_QUESTIONS} diagnosis and {N_MEDICATION_QUESTIONS} medication questions...")

# Get all diagnosis and medication queries
diagnosis_df = dataset[dataset['namespace'] == 'diagnosis_knowledge']
medication_df = dataset[dataset['namespace'] == 'medication_knowledge']

print(f"  • Available diagnosis queries: {len(diagnosis_df)}")
print(f"  • Available medication queries: {len(medication_df)}")

# Select first N from each (or all if less than N)
diagnosis_selected = diagnosis_df.head(N_DIAGNOSIS_QUESTIONS)
medication_selected = medication_df.head(N_MEDICATION_QUESTIONS)

# Combine into evaluation dataset
selected_dataset = pd.concat([diagnosis_selected, medication_selected], ignore_index=True)

print(f"\n✅ Selected {len(diagnosis_selected)} diagnosis + {len(medication_selected)} medication = {len(selected_dataset)} total queries")

# Display selected questions
print(f"\n📚 SELECTED DIAGNOSIS QUESTIONS:")
print("=" * 60)
for i, row in diagnosis_selected.iterrows():
    print(f"  {i+1}. {row['question'][:80]}...")

print(f"\n💊 SELECTED MEDICATION QUESTIONS:")
print("=" * 60)
for i, row in medication_selected.iterrows():
    print(f"  {i+1}. {row['question'][:80]}...")

# Initialize LLM evaluator
llm_evaluator = LLMEvaluator()

results = []
llm_evaluation_results = []

print("\n" + "=" * 80)
print(f"🚀 STARTING EVALUATION (Total: {len(selected_dataset)} queries)")
print("=" * 80)

for idx in tqdm(range(len(selected_dataset)), desc="Evaluating queries", unit="query", ncols=100):
    
    row = selected_dataset.iloc[idx]
    question = row["question"]
    ground_truth = row["ground_truth"]
    namespace = row["namespace"]
    
    print(f"\n{'='*60}")
    print(f"📋 QUERY #{idx+1} [{namespace.upper()}] - {idx+1}/{len(selected_dataset)}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    try:
        # Retrieve documents WITH BGE RERANKING
        docs = retrieve_with_reranking(question, namespace, k=K_RETRIEVAL, use_reranker=USE_RERANKER)
        
        if len(docs) == 0:
            raise ValueError("No documents retrieved")
        
        print(f"\n📄 Retrieved {len(docs)} documents from Pinecone")
        
        # Show retrieved document previews with rerank scores if available
        for i, doc in enumerate(docs, 1):
            preview = doc.page_content[:150].replace('\n', ' ')
            rerank_score = doc.metadata.get('rerank_score', 'N/A')
            if rerank_score != 'N/A':
                print(f"   Doc {i} (rerank: {rerank_score:.4f}): {preview}...")
            else:
                print(f"   Doc {i}: {preview}...")
        
        # Combine context with increased budget
        context = combine_context(docs, max_chars=7000)
        print(f"\n📚 Combined context length: {len(context)} characters")
        
        # Generate answer using LLM (based ONLY on context)
        print(f"\n🤖 Generating answer from context (LLM)...")
        answer = llm_evaluator.generate_answer(question, context)
        print(f"\n✅ GENERATED ANSWER:")
        print(f"{'='*60}")
        # Print answer with line wrapping
        answer_lines = answer.split('\n')
        for line in answer_lines:
            if len(line) > 80:
                words = line.split()
                current_line = ""
                for word in words:
                    if len(current_line) + len(word) + 1 <= 80:
                        current_line += (" " if current_line else "") + word
                    else:
                        print(current_line)
                        current_line = word
                if current_line:
                    print(current_line)
            else:
                print(line)
        print(f"{'='*60}")
        
        # Calculate embedding-based metrics
        doc_scores = [semantic_similarity(ground_truth, doc.page_content) for doc in docs]
        
        recall_cont = context_recall_continuous(ground_truth, docs)
        precision_cont = context_precision_continuous(ground_truth, docs)
        mrr = mrr_score(ground_truth, docs)
        ndcg = ndcg_score(ground_truth, docs)
        best_similarity = max(doc_scores) if doc_scores else 0.0
        
        prec_at_k = {k: precision_at_k(ground_truth, docs, k) for k in [1, 2, 3, 4, 5]}
        rec_at_k = {k: recall_at_k(ground_truth, docs, k) for k in [1, 2, 3, 4, 5]}
        
        # LLM evaluation
        print(f"\n🔍 Running LLM Evaluation...")
        llm_scores = llm_evaluator.evaluate_all(question, answer, ground_truth, context)
        
        faithfulness = llm_scores.get('faithfulness', {}).get('faithfulness_score', 0)
        correctness = llm_scores.get('answer_correctness', {}).get('correctness_score', 0)
        context_precision_llm = llm_scores.get('context_precision_llm', {}).get('context_precision_score', 0)
        answer_relevance = llm_scores.get('answer_relevance', {}).get('answer_relevance_score', 0)
        context_recall_llm = llm_scores.get('context_recall_llm', {}).get('context_recall_score', 0)
        
        # Store results
        llm_evaluation_results.append({
            "query_id": idx + 1,
            "namespace": namespace,
            "question": question,
            "generated_answer": answer,
            "faithfulness": faithfulness,
            "answer_correctness": correctness,
            "context_precision_llm": context_precision_llm,
            "answer_relevance": answer_relevance,
            "context_recall_llm": context_recall_llm
        })
        
        results.append({
            "query_id": idx + 1,
            "namespace": namespace,
            "question": question,
            "ground_truth": ground_truth[:500],
            "generated_answer": answer[:500],
            "recall_continuous": recall_cont,
            "precision_continuous": precision_cont,
            "mrr": mrr,
            "ndcg_graded": ndcg,
            "best_similarity": best_similarity,
            "precision@1": prec_at_k[1],
            "precision@2": prec_at_k[2],
            "precision@3": prec_at_k[3],
            "precision@4": prec_at_k[4],
            "precision@5": prec_at_k[5],
            "recall@1": rec_at_k[1],
            "recall@2": rec_at_k[2],
            "recall@3": rec_at_k[3],
            "recall@4": rec_at_k[4],
            "recall@5": rec_at_k[5],
            "faithfulness": faithfulness,
            "answer_correctness": correctness,
            "context_precision_llm": context_precision_llm,
            "answer_relevance": answer_relevance,
            "context_recall_llm": context_recall_llm,
        })
        
        print(f"\n📊 RESULTS for Query #{idx+1}:")
        print(f"   • Embedding Recall: {recall_cont:.3f}")
        print(f"   • MRR: {mrr:.3f}")
        print(f"   • Faithfulness: {faithfulness:.3f} ↑")
        print(f"   • Answer Correctness: {correctness:.3f}")
        print(f"   • Answer Relevance: {answer_relevance:.3f}")
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        results.append({
            "query_id": idx + 1,
            "namespace": namespace,
            "question": question,
            "ground_truth": ground_truth[:500],
            "generated_answer": f"ERROR: {str(e)}",
            "recall_continuous": 0, "precision_continuous": 0, "mrr": 0, "ndcg_graded": 0,
            "best_similarity": 0,
            "precision@1": 0, "precision@2": 0, "precision@3": 0, "precision@4": 0, "precision@5": 0,
            "recall@1": 0, "recall@2": 0, "recall@3": 0, "recall@4": 0, "recall@5": 0,
            "faithfulness": 0, "answer_correctness": 0, "context_precision_llm": 0,
            "answer_relevance": 0, "context_recall_llm": 0,
        })
    
    sys.stdout.flush()

# =========================================
# PROCESS RESULTS
# =========================================
print("\n" + "=" * 80)
print("📊 PROCESSING RESULTS & CREATING VISUALIZATIONS")
print("=" * 80)

results_df = pd.DataFrame(results)
llm_df = pd.DataFrame(llm_evaluation_results)

# Separate results by namespace
diag_results = results_df[results_df['namespace'] == 'diagnosis_knowledge']
med_results = results_df[results_df['namespace'] == 'medication_knowledge']

print(f"\n📈 Results Summary:")
print(f"  • Total queries evaluated: {len(results_df)}")
print(f"  • Diagnosis queries: {len(diag_results)}")
print(f"  • Medication queries: {len(med_results)}")

valid_results = results_df[results_df['best_similarity'] > 0]

if len(valid_results) > 0:
    llm_avg = {
        'avg_faithfulness': llm_df['faithfulness'].mean(),
        'avg_correctness': llm_df['answer_correctness'].mean(),
        'avg_context_precision_llm': llm_df['context_precision_llm'].mean(),
        'avg_answer_relevance': llm_df['answer_relevance'].mean(),
        'avg_context_recall_llm': llm_df['context_recall_llm'].mean(),
        'faithfulness_scores': llm_df['faithfulness'].tolist(),
        'correctness_scores': llm_df['answer_correctness'].tolist(),
        'context_precision_scores': llm_df['context_precision_llm'].tolist(),
        'answer_relevance_scores': llm_df['answer_relevance'].tolist(),
    }
    
    overall_metrics = {
        "Context Recall (continuous)": float(valid_results["recall_continuous"].mean()),
        "Context Precision (continuous)": float(valid_results["precision_continuous"].mean()),
        "MRR": float(valid_results["mrr"].mean()),
        "NDCG": float(valid_results["ndcg_graded"].mean()),
        "Precision@1": float(valid_results["precision@1"].mean()),
        "Precision@2": float(valid_results["precision@2"].mean()),
        "Precision@3": float(valid_results["precision@3"].mean()),
        "Precision@4": float(valid_results["precision@4"].mean()),
        "Precision@5": float(valid_results["precision@5"].mean()),
        "Recall@1": float(valid_results["recall@1"].mean()),
        "Recall@2": float(valid_results["recall@2"].mean()),
        "Recall@3": float(valid_results["recall@3"].mean()),
        "Recall@4": float(valid_results["recall@4"].mean()),
        "Recall@5": float(valid_results["recall@5"].mean()),
    }
    
    # Diagnosis-specific metrics
    if len(diag_results) > 0:
        diag_metrics = {
            "recall": float(diag_results["recall_continuous"].mean()),
            "precision": float(diag_results["precision_continuous"].mean()),
            "mrr": float(diag_results["mrr"].mean()),
            "faithfulness": float(diag_results["faithfulness"].mean()),
        }
    else:
        diag_metrics = {}
    
    # Medication-specific metrics
    if len(med_results) > 0:
        med_metrics = {
            "recall": float(med_results["recall_continuous"].mean()),
            "precision": float(med_results["precision_continuous"].mean()),
            "mrr": float(med_results["mrr"].mean()),
            "faithfulness": float(med_results["faithfulness"].mean()),
        }
    else:
        med_metrics = {}
    
    summary = {
        "evaluation_timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "model": f"GROQ {GROQ_MODEL}",
        "k_retrieval": K_RETRIEVAL,
        "reranker_enabled": USE_RERANKER,
        "reranker_model": BGE_RERANKER_MODEL if USE_RERANKER else "None",
        "total_queries": len(results_df),
        "diagnosis_queries": len(diag_results),
        "medication_queries": len(med_results),
        "successful_queries": len(valid_results),
        "success_rate": len(valid_results) / len(results_df),
        "overall_metrics": overall_metrics,
        "diagnosis_metrics": diag_metrics,
        "medication_metrics": med_metrics,
        "llm_metrics": llm_avg,
        "chunking_config": {
            "chunk_size": 900,
            "chunk_overlap": 150,
            "embedding_model": "abhinand/MedEmbed-base-v0.1",
            "semantic_chunker_available": SEMANTIC_AVAILABLE
        }
    }
    
    # Create plots
    print("\n📈 Generating visualizations...")
    fig = create_plots(results_df, llm_avg, summary)
    
    # Save results
    print("\n💾 Saving results...")
    rerank_suffix = "_with_reranker" if USE_RERANKER else "_no_reranker"
    results_df.to_csv(f"results_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.csv", index=False)
    llm_df.to_csv(f"llm_results_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.csv", index=False)
    
    with open(f"metrics_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.json", "w") as f:
        json.dump(summary, f, indent=4)
    
    # Print final report
    print("\n" + "=" * 80)
    print("📊 FINAL EVALUATION REPORT")
    print("=" * 80)
    
    print(f"\n✅ EVALUATION SUMMARY:")
    print(f"  • Total Queries: {summary['total_queries']} (20 Diagnosis + 20 Medication)")
    print(f"  • Diagnosis Queries: {summary['diagnosis_queries']}")
    print(f"  • Medication Queries: {summary['medication_queries']}")
    print(f"  • Success Rate: {summary['success_rate']*100:.1f}%")
    print(f"  • BGE Reranker: {'✅ ENABLED' if USE_RERANKER else '❌ DISABLED'}")
    
    print(f"\n{'='*40}")
    print("📊 OVERALL METRICS")
    print(f"{'='*40}")
    print(f"  • Recall (continuous): {overall_metrics['Context Recall (continuous)']:.4f}")
    print(f"  • Precision (continuous): {overall_metrics['Context Precision (continuous)']:.4f}")
    print(f"  • MRR: {overall_metrics['MRR']:.4f}")
    print(f"  • NDCG: {overall_metrics['NDCG']:.4f}")
    
    print(f"\n{'='*40}")
    print("📈 PRECISION@K & RECALL@K")
    print(f"{'='*40}")
    print("  k   Precision@k   Recall@k")
    for k in [1,2,3,4,5]:
        print(f"  {k}     {overall_metrics[f'Precision@{k}']:.4f}       {overall_metrics[f'Recall@{k}']:.4f}")
    
    print(f"\n{'='*40}")
    print("🤖 LLM METRICS (RAGAS)")
    print(f"{'='*40}")
    print(f"  • Faithfulness: {llm_avg['avg_faithfulness']:.4f}")
    print(f"  • Answer Correctness: {llm_avg['avg_correctness']:.4f}")
    print(f"  • Context Precision: {llm_avg['avg_context_precision_llm']:.4f}")
    print(f"  • Answer Relevance: {llm_avg['avg_answer_relevance']:.4f}")
    
    if diag_metrics:
        print(f"\n{'='*40}")
        print("🩺 DIAGNOSIS KNOWLEDGE (20 queries)")
        print(f"{'='*40}")
        print(f"  • Recall: {diag_metrics['recall']:.4f}")
        print(f"  • Precision: {diag_metrics['precision']:.4f}")
        print(f"  • MRR: {diag_metrics['mrr']:.4f}")
        print(f"  • Faithfulness: {diag_metrics['faithfulness']:.4f}")
    
    if med_metrics:
        print(f"\n{'='*40}")
        print("💊 MEDICATION KNOWLEDGE (20 queries)")
        print(f"{'='*40}")
        print(f"  • Recall: {med_metrics['recall']:.4f}")
        print(f"  • Precision: {med_metrics['precision']:.4f}")
        print(f"  • MRR: {med_metrics['mrr']:.4f}")
        print(f"  • Faithfulness: {med_metrics['faithfulness']:.4f}")
    
    # Show sample answers
    print(f"\n{'='*40}")
    print("📝 SAMPLE GENERATED ANSWERS")
    print(f"{'='*40}")
    
    # Sample from diagnosis
    diag_samples = llm_df[llm_df['namespace'] == 'diagnosis_knowledge'].head(2)
    if len(diag_samples) > 0:
        print("\n🔬 DIAGNOSIS SAMPLE ANSWERS:")
        for i, row in diag_samples.iterrows():
            print(f"\n  Q: {row['question'][:80]}...")
            print(f"  Faithfulness: {row['faithfulness']:.3f}")
            print(f"  A: {row['generated_answer'][:200]}...")
    
    # Sample from medication
    med_samples = llm_df[llm_df['namespace'] == 'medication_knowledge'].head(2)
    if len(med_samples) > 0:
        print("\n💊 MEDICATION SAMPLE ANSWERS:")
        for i, row in med_samples.iterrows():
            print(f"\n  Q: {row['question'][:80]}...")
            print(f"  Faithfulness: {row['faithfulness']:.3f}")
            print(f"  A: {row['generated_answer'][:200]}...")
    
    print("\n" + "=" * 80)
    print("✅ EVALUATION COMPLETE!")
    print("=" * 80)
    print(f"\n📁 Output files:")
    print(f"  • results_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.csv - Complete results")
    print(f"  • llm_results_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.csv - LLM scores")
    print(f"  • metrics_groq_{K_RETRIEVAL}_20x20{rerank_suffix}.json - Summary metrics")
    print(f"  • evaluation_plots.png - 9 comprehensive plots")
    
    print("\n💡 RECOMMENDED NEXT EXPERIMENTS:")
    print("  1. Use SemanticChunker with MedEmbed (already prepared)")
    print("  2. Test without reranker vs with reranker")
    print("  3. Test different chunk sizes (900, 1000, 1200)")
    print("  4. Test different context budgets (5000, 7000, 10000)")
    print("=" * 80)
    
else:
    print("❌ No successful retrievals to evaluate")

print(f"\n🏁 Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")